In [ ]:
# Llama-3.2.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""


In [ ]:
import numpy as np
from pathlib import Path

OUT_DIR = Path('/kaggle/working/Assignment1_q4'); OUT_DIR.mkdir(parents=True, exist_ok=True)
LAYERS = [0, 7, 15]
SHOW = 200          # only the first 200 
THRESHOLD = 50.0    # % cumulative energy

results = {}
for k in LAYERS:
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()
    S = torch.linalg.svdvals(q_weight.float())                    # singular values only (full SVD spectrum)
    energy = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
    min_rank = int(torch.searchsorted(energy, torch.tensor(THRESHOLD)).item()) + 1
    results[k] = dict(S=S.numpy(), energy=energy.numpy(), min_rank=min_rank)
    print(f'Layer {k:2d}: shape={tuple(q_weight.shape)}, full rank={len(S)}, '
          f'min rank for >= {THRESHOLD:.0f}% energy = {min_rank}  '
          f'(energy at that rank = {energy[min_rank-1]:.2f}%), '
          f'sigma_1={S[0]:.3f}, sigma_200={S[SHOW-1]:.3f}, sigma_min={S[-1]:.4f}')

# Summary table
import pandas as pd
table = pd.DataFrame({f'Layer {k}': [results[k]['min_rank']] for k in LAYERS},
                     index=['Minimum target rank for q_proj (>=50% energy)'])
display(table)
table.to_csv(OUT_DIR / 'q4_min_rank.csv')

# 6 charts: line (cumulative energy) + bar (singular value magnitude) per layer
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
x = np.arange(1, SHOW + 1)
for col, k in enumerate(LAYERS):
    r = results[k]
    ax = axes[0, col]
    ax.plot(x, r['energy'][:SHOW], color='#4C72B0')
    ax.axhline(THRESHOLD, ls='--', color='gray', lw=1)
    ax.axvline(r['min_rank'], ls=':', color='#C44E52', lw=1)
    ax.annotate(f"rank {r['min_rank']}", (r['min_rank'], THRESHOLD),
                xytext=(8, -14), textcoords='offset points', color='#C44E52')
    ax.set_title(f'Layer {k} q_proj: cumulative energy')
    ax.set_xlabel('Target rank'); ax.set_ylabel('Cumulative energy (%)')
    ax.set_ylim(0, 100)

    ax = axes[1, col]
    ax.bar(x, r['S'][:SHOW], width=1.0, color='#55A868')
    ax.set_title(f'Layer {k} q_proj: singular values')
    ax.set_xlabel('Singular value index'); ax.set_ylabel('Singular value magnitude')
plt.tight_layout()
plt.savefig(OUT_DIR / 'q4_all_layers.png', dpi=200)
plt.show()

# Also save each chart separately (handy for Word)
for k in LAYERS:
    r = results[k]
    f, a = plt.subplots(figsize=(5, 3.2))
    a.plot(x, r['energy'][:SHOW]); a.axhline(THRESHOLD, ls='--', color='gray', lw=1)
    a.axvline(r['min_rank'], ls=':', color='#C44E52', lw=1)
    a.set_title(f'Layer {k}: cumulative energy'); a.set_xlabel('Target rank'); a.set_ylabel('Cumulative energy (%)')
    a.set_ylim(0, 100); f.tight_layout(); f.savefig(OUT_DIR / f'q4_layer{k}_energy.png', dpi=200); plt.close(f)
    f, a = plt.subplots(figsize=(5, 3.2))
    a.bar(x, r['S'][:SHOW], width=1.0, color='#55A868')
    a.set_title(f'Layer {k}: singular values'); a.set_xlabel('Singular value index'); a.set_ylabel('Magnitude')
    f.tight_layout(); f.savefig(OUT_DIR / f'q4_layer{k}_sv.png', dpi=200); plt.close(f)
print('Saved charts to', OUT_DIR)
